# Train Multiple Versions on Kaggle & Track with W&B

**Task**: Fine-tune DistilBERT on AG News (4-class text classification)  
**Platform**: Kaggle (GPU T4 x2)  
**Tracking**: Weights & Biases (W&B)

**Prerequisites**:
- Enable GPU: Settings → Accelerator → GPU T4 x2
- Add Kaggle Secrets: `WANDB_API_KEY`, `HF_TOKEN`, `GITHUB_TOKEN`
- Dataset on HF Hub: [`YuvarajK-g25ait2054/ag-news-cleaned`](https://huggingface.co/YuvarajK-g25ait2054/ag-news-distilbert)

## Step 0: Install Dependencies

In [1]:
!pip install -q transformers datasets wandb scikit-learn accelerate 2>/dev/null && echo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/12.2 MB ? eta -:--:--

   ╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.3/12.2 MB 8.8 MB/s eta 0:00:02

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━ 8.7/12.2 MB 84.9 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━ 11.4/12.2 MB 124.9 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 76.8 MB/s eta 0:00:00


## Step 1: Load Secrets in Kaggle

API keys are stored securely using Kaggle Secrets — never hardcoded.

In [2]:
from kaggle_secrets import UserSecretsClient
import os
import wandb
from huggingface_hub import login

# Load secrets securely
secrets = UserSecretsClient()
os.environ['WANDB_API_KEY'] = secrets.get_secret('wb_key')
HF_TOKEN = secrets.get_secret('hf_key')
login(token=HF_TOKEN)
wandb.login()

# GitHub token for pushing changes
github_token = secrets.get_secret('GITHUB_TOKEN')

print("Successfully authenticated with W&B, Hugging Face, and GitHub!")

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.


wandb: Currently logged in as: g25ait2054 (g25ait2054-iit-jodhpur) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Successfully authenticated with W&B, Hugging Face, and GitHub!


## Step 2: Load Prepared Dataset & Model

In [3]:
import json
import re
import string
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# load ag_news directly - no need to upload dataset anywhere
print('Loading ag_news from HuggingFace...')
raw_dataset = load_dataset('ag_news')

# basic text cleaning
def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()
    return text

dataset = raw_dataset.map(lambda x: {'text': clean_text(x['text']), 'label': x['label']})
print(dataset)

id2label = {"0": "World", "1": "Sports", "2": "Business", "3": "Sci/Tech"}
label2id = {v: int(k) for k, v in id2label.items()}
num_labels = len(id2label)

print(f'Labels: {id2label}')
print(f'Number of classes: {num_labels}')

Loading ag_news from HuggingFace...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})
Labels: {'0': 'World', '1': 'Sports', '2': 'Business', '3': 'Sci/Tech'}
Number of classes: 4


In [4]:
# Preview the data
print(f"Train samples: {len(dataset['train'])}")
print(f"Test samples:  {len(dataset['test'])}")
dataset["train"].to_pandas().head()

Train samples: 120000
Test samples:  7600


,text,label
0,wall st bears claw back into the black reuters...,2
1,carlyle looks toward commercial aerospace reut...,2
2,oil and economy cloud stocks outlook reuters r...,2
3,iraq halts oil exports from main southern pipe...,2
4,oil prices soar to alltime record posing new m...,2


In [5]:
# Assign train/test splits
train_dataset = dataset["train"]
test_dataset = dataset["test"]

print(f"Train dataset: {train_dataset}")
print(f"Test dataset:  {test_dataset}")

Train dataset: Dataset({
    features: ['text', 'label'],
    num_rows: 120000
})
Test dataset:  Dataset({
    features: ['text', 'label'],
    num_rows: 7600
})


In [6]:
# Load tokenizer
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Tokenize datasets
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

# Set format for PyTorch
train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])

print("Tokenization complete!")
print(f"Sample features: {train_dataset[0].keys()}")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

Tokenization complete!
Sample features: dict_keys(['label', 'input_ids', 'attention_mask'])


## Step 3: Define Metrics

In [7]:
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted"),
    }

---
## Step 4: Training Version 1

**Hyperparameters (v1)**:
- Learning rate: `2e-5`
- Batch size: `16`
- Epochs: `3`
- Weight decay: `0.01`

In [8]:
from transformers import TrainingArguments, Trainer

# Initialise W&B run for Version 1
wandb.init(
    project="mlops-assignment3",
    name="run-ver1",
    config={
        "model": model_name,
        "epochs": 3,
        "batch_size": 16,
        "learning_rate": 2e-5,
        "weight_decay": 0.01,
        "max_length": 128,
        "version": "v1",
        "platform": "Kaggle",
    }
)

print("W&B run initialised: run-ver1")

wandb: Tracking run with wandb version 0.25.1


wandb: Run data is saved locally in /kaggle/working/wandb/run-20260608_171854-7ph9mtnb
wandb: Run `wandb offline` to turn off syncing.


wandb: Syncing run run-ver1


wandb: ⭐️ View project at https://wandb.ai/g25ait2054-iit-jodhpur/mlops-assignment3


wandb: 🚀 View run at https://wandb.ai/g25ait2054-iit-jodhpur/mlops-assignment3/runs/7ph9mtnb


W&B run initialised: run-ver1


In [9]:
# Load fresh model for v1
model_v1 = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

# Training arguments for v1
training_args_v1 = TrainingArguments(
    output_dir="./results-v1",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="wandb",
    run_name="run-ver1",
    logging_steps=100,
)

# Create Trainer
trainer_v1 = Trainer(
    model=model_v1,
    args=training_args_v1,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

# Train
print("Starting training v1...")
trainer_v1.train()

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Starting training v1...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].


There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=11250, training_loss=0.3340312205844455, metrics={'train_runtime': 2558.0466, 'train_samples_per_second': 140.732, 'train_steps_per_second': 4.398, 'total_flos': 1.192249110528e+16, 'train_loss': 0.3340312205844455, 'epoch': 3.0})

In [10]:
# Evaluate v1
results_v1 = trainer_v1.evaluate()
print(f"\n=== Version 1 Results ===")
print(f"  Accuracy: {results_v1['eval_accuracy']:.4f}")
print(f"  F1 Score: {results_v1['eval_f1']:.4f}")
print(f"  Eval Loss: {results_v1['eval_loss']:.4f}")

# Finish W&B run
wandb.finish()
print("\nW&B run-ver1 finished.")

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


wandb: updating run metadata



=== Version 1 Results ===
  Accuracy: 0.9434
  F1 Score: 0.9434
  Eval Loss: 0.3679


wandb: uploading wandb-summary.json; uploading config.yaml; uploading output.log


wandb: 
wandb: Run history:
wandb:           eval/accuracy ▁█▇█
wandb:                 eval/f1 ▁█▇█
wandb:               eval/loss ▄▁█▁
wandb:            eval/runtime ▄▁█▂
wandb: eval/samples_per_second ▅█▁▇
wandb:   eval/steps_per_second ▅█▁▇
wandb:             train/epoch ▁▁▁▁▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇███
wandb:       train/global_step ▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▇▇▇▇▇▇▇█████
wandb:         train/grad_norm ▅▄▄▃▂▃▄▂▅▅▃▂▂▇▁▃▃▆▄▅▅▁▅▂▃▂▃▁▃▄▁█▄▂▅▂▂▅▂▄
wandb:     train/learning_rate ████▇▇▇▇▇▇▇▆▆▆▆▆▆▆▅▅▅▅▅▄▄▄▄▄▄▃▃▃▃▂▂▂▂▁▁▁
wandb:                      +1 ...
wandb: 
wandb: Run summary:
wandb:           eval/accuracy 0.94342
wandb:                 eval/f1 0.94342
wandb:               eval/loss 0.36794
wandb:            eval/runtime 15.0542
wandb: eval/samples_per_second 504.842
wandb:   eval/steps_per_second 7.905
wandb:              total_flos 1.192249110528e+16
wandb:             train/epoch 3
wandb:       train/global_step 11250
wandb:         train/grad_norm 9.50418
wandb:        

wandb: 🚀 View run run-ver1 at: https://wandb.ai/g25ait2054-iit-jodhpur/mlops-assignment3/runs/7ph9mtnb
wandb: ⭐️ View project at: https://wandb.ai/g25ait2054-iit-jodhpur/mlops-assignment3
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)


wandb: Find logs at: ./wandb/run-20260608_171854-7ph9mtnb/logs



W&B run-ver1 finished.


---
## Step 5: Training Version 2

**Hyperparameters (v2)** — changed learning rate and batch size:
- Learning rate: `5e-5` (increased from 2e-5)
- Batch size: `64` (increased from 16)
- Epochs: `5`
- Weight decay: `0.01

In [11]:
# Initialise W&B run for Version 2
wandb.init(
    project="mlops-assignment3",
    name="run-ver2",
    config={
        "model": model_name,
        "epochs": 5, # Please take: 4, 5, 7,  
        "batch_size": 64,
        "learning_rate": 5e-5,
        "weight_decay": 0.01,
        "max_length": 128,
        "version": "v2",
        "platform": "Kaggle",
    }
)

print("W&B run initialised: run-ver2")

wandb: setting up run rnjku4vv


wandb: Tracking run with wandb version 0.25.1


wandb: Run data is saved locally in /kaggle/working/wandb/run-20260608_180153-rnjku4vv
wandb: Run `wandb offline` to turn off syncing.


wandb: Syncing run run-ver2


wandb: ⭐️ View project at https://wandb.ai/g25ait2054-iit-jodhpur/mlops-assignment3


wandb: 🚀 View run at https://wandb.ai/g25ait2054-iit-jodhpur/mlops-assignment3/runs/rnjku4vv


W&B run initialised: run-ver2


In [12]:
# Load fresh model for v2 (start from pretrained, not from v1 checkpoint)
model_v2 = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

# Training arguments for v2
training_args_v2 = TrainingArguments(
    output_dir="./results-v2",
    num_train_epochs=5,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    learning_rate=5e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="wandb",
    run_name="run-ver2",
    logging_steps=100,
)

# Create Trainer
trainer_v2 = Trainer(
    model=model_v2,
    args=training_args_v2,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

# Train
print("Starting training v2...")
trainer_v2.train()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Starting training v2...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.392430,0.385509,0.933816,0.933671
2,0.281550,0.357545,0.941053,0.941074
3,0.170281,0.412348,0.938158,0.938149
4,0.104768,0.461349,0.938947,0.938946
5,0.057523,0.555514,0.937105,0.937070


...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].


There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=4690, training_loss=0.22223039903620412, metrics={'train_runtime': 3279.7589, 'train_samples_per_second': 182.94, 'train_steps_per_second': 1.43, 'total_flos': 1.98708185088e+16, 'train_loss': 0.22223039903620412, 'epoch': 5.0})

In [13]:
# Evaluate v2
results_v2 = trainer_v2.evaluate()
print(f"\n=== Version 2 Results ===")
print(f"  Accuracy: {results_v2['eval_accuracy']:.4f}")
print(f"  F1 Score: {results_v2['eval_f1']:.4f}")
print(f"  Eval Loss: {results_v2['eval_loss']:.4f}")

# Finish W&B run
wandb.finish()
print("\nW&B run-ver2 finished.")

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


wandb: updating run metadata



=== Version 2 Results ===
  Accuracy: 0.9412
  F1 Score: 0.9412
  Eval Loss: 0.3575


wandb: uploading history steps 52-52, summary, console lines 32-36


wandb: 
wandb: Run history:
wandb:           eval/accuracy ▁█▅▆▄█
wandb:                 eval/f1 ▁█▅▆▄█
wandb:               eval/loss ▂▁▃▅█▁
wandb:            eval/runtime ▆█▇▇█▁
wandb: eval/samples_per_second ▃▁▂▂▁█
wandb:   eval/steps_per_second ▃▁▂▂▁█
wandb:             train/epoch ▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇████
wandb:       train/global_step ▁▁▁▁▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇████
wandb:         train/grad_norm ▃▃▂▂▂▂▂▃▁▃▂▄▂▃▃▂▁▃▂▃▂▂▃▃▄▄▄▅▃▃▃▃▁▃▁▂▂▃▁█
wandb:     train/learning_rate ████▇▇▇▇▇▆▆▆▆▆▅▅▅▅▅▅▄▄▄▄▄▄▃▃▃▃▃▂▂▂▂▂▁▁▁▁
wandb:                      +1 ...
wandb: 
wandb: Run summary:
wandb:           eval/accuracy 0.94118
wandb:                 eval/f1 0.94121
wandb:               eval/loss 0.3575
wandb:            eval/runtime 14.4821
wandb: eval/samples_per_second 524.786
wandb:   eval/steps_per_second 4.143
wandb:              total_flos 1.98708185088e+16
wandb:             train/epoch 5
wandb:       train/global_step 4690
wandb:         train/grad_norm 7.34433
wandb

wandb: 🚀 View run run-ver2 at: https://wandb.ai/g25ait2054-iit-jodhpur/mlops-assignment3/runs/rnjku4vv
wandb: ⭐️ View project at: https://wandb.ai/g25ait2054-iit-jodhpur/mlops-assignment3
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)


wandb: Find logs at: ./wandb/run-20260608_180153-rnjku4vv/logs



W&B run-ver2 finished.


---
## Step 6: Compare Results

In [14]:
# Side-by-side comparison
print("=" * 60)
print("EXPERIMENT COMPARISON")
print("=" * 60)
print(f"{'Metric':<20} {'Version 1':<15} {'Version 2':<15}")
print("-" * 50)
print(f"{'Learning Rate':<20} {'2e-5':<15} {'5e-5':<15}")
print(f"{'Batch Size':<20} {'16':<15} {'32':<15}")
print(f"{'Epochs':<20} {'3':<15} {'3':<15}")
print("-" * 50)
print(f"{'Accuracy':<20} {results_v1['eval_accuracy']:<15.4f} {results_v2['eval_accuracy']:<15.4f}")
print(f"{'F1 (weighted)':<20} {results_v1['eval_f1']:<15.4f} {results_v2['eval_f1']:<15.4f}")
print(f"{'Eval Loss':<20} {results_v1['eval_loss']:<15.4f} {results_v2['eval_loss']:<15.4f}")
print("=" * 60)

# Determine best version
best = "v1" if results_v1['eval_f1'] > results_v2['eval_f1'] else "v2"
print(f"\nBest version: {best}")
print(f"\nCheck your W&B dashboard at: https://wandb.ai/ (project: mlops-assignment3)")

EXPERIMENT COMPARISON
Metric               Version 1       Version 2      
--------------------------------------------------
Learning Rate        2e-5            5e-5           
Batch Size           16              32             
Epochs               3               3              
--------------------------------------------------
Accuracy             0.9434          0.9412         
F1 (weighted)        0.9434          0.9412         
Eval Loss            0.3679          0.3575         

Best version: v1

Check your W&B dashboard at: https://wandb.ai/ (project: mlops-assignment3)


---
## Step 7: Push the Trained Model to Hugging Face Hub

Push the **best** trained model and tokenizer to a public Hugging Face repository so GitHub Actions can load it for inference.

> **Important**: Set the repository visibility to **Public** on Hugging Face before submitting.

In [15]:
# ---- CHANGE THIS to your Hugging Face username ----
HF_USERNAME = "YuvarajK-g25ait2054"
HF_REPO_NAME = "ag-news-distilbert"
HF_REPO_ID = f"{HF_USERNAME}/{HF_REPO_NAME}"

# Select the best model based on F1 score
if results_v1["eval_f1"] >= results_v2["eval_f1"]:
    best_trainer = trainer_v1
    best_version = "v1"
else:
    best_trainer = trainer_v2
    best_version = "v2"

print(f"Best version: {best_version}")
print(f"Pushing to: https://huggingface.co/{HF_REPO_ID}")

Best version: v1
Pushing to: https://huggingface.co/YuvarajK-g25ait2054/ag-news-distilbert


In [16]:
# Push model and tokenizer to Hugging Face Hub
best_trainer.model.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
tokenizer.push_to_hub(HF_REPO_ID, token=HF_TOKEN)

print(f"Model and tokenizer pushed to: https://huggingface.co/{HF_REPO_ID}")

README.md: 0.00B [00:00, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


Model and tokenizer pushed to: https://huggingface.co/YuvarajK-g25ait2054/ag-news-distilbert


In [17]:
# Log the Hugging Face model URL into W&B run summary
hf_model_url = f"https://huggingface.co/{HF_REPO_ID}"

wandb.init(
    project="mlops-assignment3",
    name=f"push-model-{best_version}",
    config={"best_version": best_version, "hf_repo": HF_REPO_ID}
)
wandb.run.summary["huggingface_model"] = hf_model_url
wandb.run.summary["best_version"] = best_version
wandb.finish()

print(f"Logged HF model URL to W&B summary: {hf_model_url}")

wandb: Tracking run with wandb version 0.25.1


wandb: Run data is saved locally in /kaggle/working/wandb/run-20260608_185659-xmwn20m2
wandb: Run `wandb offline` to turn off syncing.


wandb: Syncing run push-model-v1


wandb: ⭐️ View project at https://wandb.ai/g25ait2054-iit-jodhpur/mlops-assignment3


wandb: 🚀 View run at https://wandb.ai/g25ait2054-iit-jodhpur/mlops-assignment3/runs/xmwn20m2


...

wandb: uploading config.yaml; uploading wandb-metadata.json; uploading wandb-summary.json


wandb: 
wandb: Run summary:
wandb:      best_version v1
wandb: huggingface_model https://huggingface....
wandb: 


wandb: 🚀 View run push-model-v1 at: https://wandb.ai/g25ait2054-iit-jodhpur/mlops-assignment3/runs/xmwn20m2
wandb: ⭐️ View project at: https://wandb.ai/g25ait2054-iit-jodhpur/mlops-assignment3
wandb: Synced 4 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)


wandb: Find logs at: ./wandb/run-20260608_185659-xmwn20m2/logs


Logged HF model URL to W&B summary: https://huggingface.co/YuvarajK-g25ait2054/ag-news-distilbert


---
## Step 8: Push Notebook & Results to GitHub

Clone the repo, copy the notebook, commit and push to the `ritesh_dev` branch.

In [ ]:
import shutil
import os

# ── GitHub Config ──────────────────────────────────────────────────────────
GITHUB_USERNAME = "riteshmaury-iitj"
REPO_NAME = "group13-assignment-mlops"
BRANCH = "develop"
REPO_PATH = f"/kaggle/working/{REPO_NAME}"

# ── Clone or update repo ──────────────────────────────────────────────────
if os.path.exists(REPO_PATH):
    shutil.rmtree(REPO_PATH)
    print("Removed existing repo folder.")

!git clone https://{github_token}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git {REPO_PATH}

# Checkout branch (create if it doesn't exist)
!cd {REPO_PATH} && git checkout {BRANCH} 2>/dev/null || git checkout -b {BRANCH}

# ── Configure git ──────────────────────────────────────────────────────────
!git -C {REPO_PATH} config user.email "g25ait2054@iitj.ac.in"
!git -C {REPO_PATH} config user.name "{GITHUB_USERNAME}"

# ── Copy notebook and files to repo ────────────────────────────────────────
import glob

# Copy notebooks
for nb in glob.glob("/kaggle/working/*.ipynb"):
    shutil.copy(nb, REPO_PATH)
    print(f"  Copied: {os.path.basename(nb)}")

# Copy id2label.json if it exists
for f in ["id2label.json"]:
    src = f"/kaggle/working/{f}"
    if os.path.exists(src):
        shutil.copy(src, REPO_PATH)
        print(f"  Copied: {f}")

# ── Commit and push ───────────────────────────────────────────────────────
!cd {REPO_PATH} && git add -A && git status
!cd {REPO_PATH} && git commit -m "Add training notebook with W&B tracking and HF model push"
!cd {REPO_PATH} && git push origin {BRANCH}

print(f"\nPushed to: https://github.com/{GITHUB_USERNAME}/{REPO_NAME}/tree/{BRANCH}")

---
## Notes for Report

1. **Screenshot**: Go to your W&B dashboard → project `mlops-assignment3` → select both runs → take a screenshot showing training loss, validation loss, accuracy, and F1 curves side by side.

2. **Hyperparameter differences**:
   - v1: lr=2e-5, batch_size=16
   - v2: lr=5e-5, batch_size=32

3. **All logged metrics**: training_loss, eval_loss, eval_accuracy, eval_f1 (logged every epoch via `eval_strategy='epoch'`)

4. **Kaggle Secrets used**: `WANDB_API_KEY`, `HF_TOKEN`, `GITHUB_TOKEN` — no tokens hardcoded in the notebook.

5. **Dataset source**: [YuvarajK-g25ait2054/ag-news-cleaned](https://huggingface.co/datasets/YuvarajK-g25ait2054/ag-news-cleaned)